# 04 — Length-controlled evaluation and sub-word tokenizer fertility

Re-scores the released predictions on controlled sub-populations. No retraining: the
files in `results/predictions/` are row-aligned with `data/cleaned/test_cleaned.json`,
which section 2 asserts before anything else runs.

- Input: `data/cleaned/*.json`, `results/predictions/preds_*.csv`
- Output: `results_r2/table_A0..A2`, `table_A6..A8` (CSV), `fig_length_control.png`
- Runtime: ~5 min, CPU (section 9 downloads four tokenizers)
- Reported in: paper Sections VI-B and VI-D

## 1. Config

In [ ]:
# CONFIG — edit these two paths only
REPO_DIR    = '.'              # root of kazakh-ai-text-detection
RESULTS_DIR = './results_r2'   # where new tables/figures are written

SEED     = 42
BIN_WIDTH = 5     # width (in words) of the matching strata
N_BOOT    = 1000  # bootstrap resamples
SEQ_LEN   = 256   # the token budget used in notebook 03

# --- Colab convenience -------------------------------------------------
IN_COLAB = False
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # EDIT: point at your copy of the repository on Drive
    REPO_DIR    = '/content/drive/MyDrive/kazakh-ai-text-detection'
    RESULTS_DIR = '/content/drive/MyDrive/kazakh-ai-text-detection/results_r2'

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'sentencepiece', 'scikit-learn', 'pandas',
                'matplotlib', 'seaborn'], check=False)
print('REPO_DIR   =', REPO_DIR)
print('RESULTS_DIR=', RESULTS_DIR)

# Auto-load the shared paths written by notebook 00_setup_and_check.
# Run notebook 00 once and you never edit a path in any notebook again.
# If r2_config.json is absent, the values set above are used unchanged.
try:
    import json as _json
    from pathlib import Path as _Path
    _cfg_path = _Path('/content/drive/MyDrive/r2_config.json')
    if _cfg_path.exists():
        _cfg = _json.load(open(_cfg_path))
        REPO_DIR = _cfg['REPO_DIR']
        RESULTS_DIR = _cfg.get('RESULTS_DIR', RESULTS_DIR)
        print('Paths loaded from r2_config.json')
        print('  REPO_DIR   =', REPO_DIR)
        print('  RESULTS_DIR=', RESULTS_DIR)
    else:
        print('r2_config.json not found -- using the paths set above. '
              'Run 00_setup_and_check.ipynb to generate it.')
except Exception as _e:
    print('Could not load r2_config.json (%s: %s) -- using the paths set above.'
          % (type(_e).__name__, _e))


## 2. Load data and predictions

In [ ]:
import json, os, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, accuracy_score
warnings.filterwarnings('ignore')

REPO = Path(REPO_DIR); OUT = Path(RESULTS_DIR); OUT.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(SEED)

ID2LABEL = {0: 'Human', 1: 'AI-Generated', 2: 'AI-Obfuscated'}
MODEL_FILES = {
    'mDeBERTa-v3 (base)':  'preds_mDeBERTa-v3_base.csv',
    'XLM-R (base)':        'preds_XLM-R_base.csv',
    'mBERT (cased)':       'preds_mBERT_cased.csv',
    'DistilmBERT':         'preds_DistilmBERT.csv',
    'TF-IDF + LogReg':     'preds_TF-IDF_+_LogReg.csv',
}

def load_split(name):
    df = pd.DataFrame(json.load(open(REPO / f'data/cleaned/{name}_cleaned.json', encoding='utf-8')))
    df['text']  = df['text'].astype(str)
    df['label'] = df['label'].astype(int)
    df['split'] = name
    df['n_words'] = df['text'].str.split().str.len()
    return df

train, dev, test = load_split('train'), load_split('dev'), load_split('test')
y_test = test['label'].to_numpy()

# --- THE CRITICAL PRECONDITION ----------------------------------------
# Every downstream result depends on prediction rows matching test rows.
preds = {}
for name, fname in MODEL_FILES.items():
    p = pd.read_csv(REPO / 'results/predictions' / fname)
    assert len(p) == len(test), f'{name}: {len(p)} rows vs {len(test)} test rows'
    assert (p['y_true'].to_numpy() == y_test).all(), (
        f'{name}: prediction rows are NOT aligned with test_cleaned.json. '
        'Do not trust any result in this notebook until this is resolved.')
    preds[name] = p['y_pred'].to_numpy()

print('Alignment verified for all %d prediction files.' % len(preds))
print('Test size:', len(test), '| class counts:', dict(test['label'].value_counts().sort_index()))


## 3. Build the length-matched subset (5-word strata, equal counts per class)

In [ ]:
test = test.copy()
test['bin'] = (test['n_words'] // BIN_WIDTH).astype(int)

keep_idx, strata_rows = [], []
for b, g in test.groupby('bin'):
    counts = g['label'].value_counts()
    if len(counts) < 3:
        continue                      # stratum cannot be matched across all classes
    k = int(counts.min())
    strata_rows.append({'bin_words': f'{b*BIN_WIDTH}-{b*BIN_WIDTH+BIN_WIDTH-1}',
                        'per_class_kept': k,
                        **{f'n_{ID2LABEL[c]}': int(counts.get(c, 0)) for c in range(3)}})
    for c in range(3):
        pool = g.index[g['label'] == c].to_numpy()
        keep_idx.extend(rng.choice(pool, size=k, replace=False).tolist())

keep_idx = np.array(sorted(keep_idx))
lm = test.loc[keep_idx]
pd.DataFrame(strata_rows).to_csv(OUT / 'lengthmatched_strata.csv', index=False)

print(f'Length-matched subset: n = {len(keep_idx)}')
print('Per class:', {ID2LABEL[c]: int((lm.label == c).sum()) for c in range(3)})
print('Word range:', int(lm.n_words.min()), '-', int(lm.n_words.max()))
print('Median words per class:',
      {ID2LABEL[c]: float(lm.loc[lm.label == c, 'n_words'].median()) for c in range(3)})

# sanity: the three class-conditional length distributions must now coincide
assert len({tuple(sorted(lm.loc[lm.label == c, 'n_words'])) and
            float(lm.loc[lm.label == c, 'n_words'].median()) for c in range(3)}) == 1, \
    'Medians differ — matching failed'


## 4. Re-score every model on the matched subset

In [ ]:
def boot_ci(y_true, y_pred, n=N_BOOT, seed=SEED):
    r = np.random.default_rng(seed)
    idx = np.arange(len(y_true)); vals = []
    for _ in range(n):
        s = r.choice(idx, size=len(idx), replace=True)
        if len(np.unique(y_true[s])) < 2:
            continue
        vals.append(f1_score(y_true[s], y_pred[s], average='macro'))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

rows = []
for m, yp in preds.items():
    full = f1_score(y_test, yp, average='macro')
    mat  = f1_score(y_test[keep_idx], yp[keep_idx], average='macro')
    lo, hi = boot_ci(y_test[keep_idx], yp[keep_idx])
    per = f1_score(y_test[keep_idx], yp[keep_idx], average=None, labels=[0, 1, 2])
    per_full = f1_score(y_test, yp, average=None, labels=[0, 1, 2])
    rows.append({'model': m,
                 'macro_f1_full': round(full, 4),
                 'macro_f1_lengthmatched': round(mat, 4),
                 'delta': round(mat - full, 4),
                 'ci_low': round(lo, 4), 'ci_high': round(hi, 4),
                 'acc_lengthmatched': round(accuracy_score(y_test[keep_idx], yp[keep_idx]), 4),
                 'f1_Human': round(per[0], 4),
                 'f1_AI-Generated': round(per[1], 4),
                 'f1_AI-Obfuscated': round(per[2], 4),
                 'f1_AI-Obfuscated_FULL': round(per_full[2], 4)})

tblA1 = pd.DataFrame(rows)
tblA1.to_csv(OUT / 'table_A1_length_matched.csv', index=False)
print(tblA1.to_string(index=False))


## 5. Word-count-only control

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

Xtr = train[['n_words']].to_numpy(float); ytr = train['label'].to_numpy()
Xdv = dev[['n_words']].to_numpy(float);   ydv = dev['label'].to_numpy()
Xte = test[['n_words']].to_numpy(float)

best = None
for C in [0.05, 0.25, 1, 5, 25]:
    pipe = make_pipeline(StandardScaler(),
                         LogisticRegression(max_iter=4000, class_weight='balanced', C=C))
    pipe.fit(Xtr, ytr)
    s = f1_score(ydv, pipe.predict(Xdv), average='macro')
    if best is None or s > best[0]:
        best = (s, C, pipe)

yp_len = best[2].predict(Xte)
len_full = f1_score(y_test, yp_len, average='macro')
len_mat  = f1_score(y_test[keep_idx], yp_len[keep_idx], average='macro')
print(f'Word-count-only control  (C={best[1]}, dev macro-F1={best[0]:.4f})')
print(f'  full test        macro-F1 = {len_full:.4f}')
print(f'  length-matched   macro-F1 = {len_mat:.4f}')
print('\nIf the second number is near chance, length alone does not solve the task.')

pd.DataFrame([{'system': 'word-count-only', 'macro_f1_full': round(len_full, 4),
               'macro_f1_lengthmatched': round(len_mat, 4), 'C': best[1]}]
             ).to_csv(OUT / 'table_A1b_length_only_control.csv', index=False)


## 6. Figure — full vs length-matched macro-F1

In [ ]:
# ---- Figure: full vs length-matched macro-F1, and the obfuscated-class F1 ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
x = np.arange(len(tblA1)); w = 0.38

ax = axes[0]
ax.bar(x - w/2, tblA1['macro_f1_full'], w, label='Full test (n=900)')
ax.bar(x + w/2, tblA1['macro_f1_lengthmatched'], w, label=f'Length-matched (n={len(keep_idx)})')
ax.set_xticks(x); ax.set_xticklabels(tblA1['model'], rotation=25, ha='right', fontsize=8)
ax.set_ylabel('Macro-F1'); ax.set_ylim(0, 1.0)
ax.set_title('Macro-F1 before and after length matching'); ax.legend(fontsize=8)
ax.grid(axis='y', alpha=.3)

ax = axes[1]
ax.bar(x - w/2, tblA1['f1_AI-Obfuscated_FULL'], w, label='Full test')
ax.bar(x + w/2, tblA1['f1_AI-Obfuscated'], w, label='Length-matched')
ax.set_xticks(x); ax.set_xticklabels(tblA1['model'], rotation=25, ha='right', fontsize=8)
ax.set_ylabel('F1 (AI-Obfuscated)'); ax.set_ylim(0, 1.0)
ax.set_title('AI-Obfuscated per-class F1 — the shortcut test'); ax.legend(fontsize=8)
ax.grid(axis='y', alpha=.3)

plt.tight_layout(); plt.savefig(OUT / 'fig_A1_length_matched.png', dpi=200)
plt.show()


## 7. Length-bucket stratification

In [ ]:
def bucket(w):
    return '<50' if w < 50 else ('50-150' if w <= 150 else '>150')

test['bucket'] = test['n_words'].map(bucket)
BUCKETS = ['<50', '50-150', '>150']

rows = []
for b in BUCKETS:
    sel = test.index[test['bucket'] == b].to_numpy()
    present = sorted(set(y_test[sel].tolist()))
    for m, yp in preds.items():
        per = f1_score(y_test[sel], yp[sel], average=None, labels=present)
        rows.append({'bucket': b, 'n': len(sel),
                     'classes_present': '/'.join(ID2LABEL[c] for c in present),
                     'model': m,
                     'macro_f1': round(f1_score(y_test[sel], yp[sel], average='macro', labels=present), 4),
                     'accuracy': round(accuracy_score(y_test[sel], yp[sel]), 4),
                     **{f'f1_{ID2LABEL[c]}': round(v, 4) for c, v in zip(present, per)}})

tblA2 = pd.DataFrame(rows)
tblA2.to_csv(OUT / 'table_A2_length_buckets.csv', index=False)

print('Class counts per bucket (report these alongside the F1 values):')
print(pd.crosstab(test['bucket'], test['label'].map(ID2LABEL)).reindex(BUCKETS).to_string())
print()
print(tblA2.pivot(index='model', columns='bucket', values='macro_f1')[BUCKETS].to_string())


## 8. Figure — class-conditional length distributions

In [ ]:
# ---- Figure: class-conditional length distributions, before and after matching ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=False)
for ax, (d, title) in zip(axes, [(test, 'Full test set'),
                                 (lm, f'Length-matched subset (n={len(lm)})')]):
    for c in range(3):
        ax.hist(d.loc[d.label == c, 'n_words'], bins=30, alpha=.55, label=ID2LABEL[c])
    ax.set_xlabel('words'); ax.set_ylabel('documents'); ax.set_title(title)
    ax.legend(fontsize=8); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT / 'fig_A2_length_distributions.png', dpi=200)
plt.show()


## 9. Sub-word fertility and truncation at 256 tokens

In [ ]:
from transformers import AutoTokenizer

TOKENIZERS = {
    'XLM-R (base)':       'xlm-roberta-base',
    'mDeBERTa-v3 (base)': 'microsoft/mdeberta-v3-base',
    'mBERT (cased)':      'bert-base-multilingual-cased',
    'DistilmBERT':        'distilbert-base-multilingual-cased',
}

fert_rows, trunc_rows, failures = [], [], []
words_all = test['n_words'].to_numpy()
labels_all = test['label'].to_numpy()

for name, hf_id in TOKENIZERS.items():
    try:
        tk = AutoTokenizer.from_pretrained(hf_id)
    except Exception as e:
        failures.append((name, hf_id, f'{type(e).__name__}: {e}'))
        print(f'[FAILED] {name} ({hf_id}) -> {type(e).__name__}')
        continue

    ntok = np.array([len(tk.encode(t, add_special_tokens=True, truncation=False))
                     for t in test['text']])
    fert = ntok / np.maximum(words_all, 1)

    for c in list(range(3)) + ['ALL']:
        sel = np.ones(len(test), bool) if c == 'ALL' else (labels_all == c)
        cname = 'ALL' if c == 'ALL' else ID2LABEL[c]
        fert_rows.append({
            'tokenizer': name, 'class': cname,
            'median_words': float(np.median(words_all[sel])),
            'median_subtokens': float(np.median(ntok[sel])),
            'mean_fertility': round(float(fert[sel].mean()), 3),
            'median_fertility': round(float(np.median(fert[sel])), 3),
            'p95_subtokens': float(np.percentile(ntok[sel], 95)),
            'max_subtokens': int(ntok[sel].max()),
        })
        trunc_rows.append({
            'tokenizer': name, 'class': cname,
            f'pct_truncated_at_{SEQ_LEN}': round(100 * float((ntok[sel] > SEQ_LEN).mean()), 2),
            'mean_retained_fraction': round(
                float(np.minimum(ntok[sel], SEQ_LEN).sum() / ntok[sel].sum()), 4),
        })

if failures:
    print('\n*** Some tokenizers could not be downloaded. ***')
    for n, i, e in failures:
        print(f'  {n:22s} {i:38s} {e}')
    print('Run this part on Colab, or set HF_TOKEN if a model is gated.')

fert_df  = pd.DataFrame(fert_rows)
trunc_df = pd.DataFrame(trunc_rows)
fert_df.to_csv(OUT / 'table_A6_tokenizer_fertility.csv', index=False)
trunc_df.to_csv(OUT / 'table_A7_truncation.csv', index=False)
print(fert_df.to_string(index=False)); print()
print(trunc_df.to_string(index=False))


## 10. Word-count tail

In [ ]:
# Word-count tail — computable without any tokenizer, useful context for the paper
allsp = pd.concat([train, dev, test], ignore_index=True)
rows = []
for c in range(3):
    s = allsp.loc[allsp.label == c, 'n_words']
    rows.append({'class': ID2LABEL[c], 'n': len(s), 'median': float(s.median()),
                 'max': int(s.max()),
                 **{f'pct_gt_{t}_words': round(100 * float((s > t).mean()), 2)
                    for t in [85, 100, 128, 170, 256]}})
tail = pd.DataFrame(rows)
tail.to_csv(OUT / 'table_A8_wordcount_tail.csv', index=False)
print(tail.to_string(index=False))
print('\nNote: the AI-obfuscated maximum is far below the token budget, so no document of that '
      'class can ever be truncated. That is why Part A is the decisive analysis, not this table.')
